# Anillo Obsidian Geochemical Sourcing

Compare the study samples with known obsidian source chemistry using a biplot and a ternary diagram. The notebook reads the two project CSVs from `data/` automatically; it does not require file uploads, Google Sheets, or map selection.

Run all cells from the repository directory. Use the axis dropdowns to choose elements for each plot and the Show Labels button to toggle study sample labels on both plots.

Ellipses show estimated 95% bivariate normal data spread for each group, not confidence intervals for the mean. They are calculated from each group's sample covariance and do not necessarily contain exactly 95% of observed samples.

In [11]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS

ELEMENTS = ["Rb", "Sr", "Zr"]


In [12]:
# Locate the checked-in data directory when Jupyter starts in the repository,
# its parent workspace, or the notebooks folder.
_search_roots = [Path.cwd(), *Path.cwd().parents]
_data_candidates = [root / "data" for root in _search_roots]
_data_candidates.extend(root / "obsidian-source-anillo" / "data" for root in _search_roots)
DATA_DIR = next(
    (path for path in _data_candidates
     if (path / "KRA21_sources.csv").is_file()
     and (path / "study_samples.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate data/KRA21_sources.csv and data/study_samples.csv. "
        "Run Jupyter from this project, its notebooks folder, or the parent workspace."
    )

BRUKER_SUFFIX = re.compile(r"\s*(?:K[ab][12]?|L[ab][12]?|M[ab][12]?)\s*$", re.IGNORECASE)


def _normalize_column_name(name):
    name = BRUKER_SUFFIX.sub("", str(name).strip())
    return re.sub(r"\s+", "_", name).strip("_")


def _load_geochemistry(path, sample_column):
    frame = pd.read_csv(path)
    frame.columns = [_normalize_column_name(column) for column in frame.columns]
    if sample_column != "Sample" and sample_column in frame.columns:
        frame = frame.rename(columns={sample_column: "Sample"})

    required = {"Sample", "Group", *ELEMENTS}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    for column in frame.columns:
        if column not in {"Sample", "Group"}:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    before = len(frame)
    frame = frame.dropna(subset=["Sample", "Group", *ELEMENTS]).copy()
    if frame.empty:
        raise ValueError(f"{path.name} has no rows with usable sample, group, Rb, Sr, and Zr values")
    if len(frame) != before:
        print(f"{path.name}: excluded {before - len(frame)} incomplete row(s).")
    return frame.reset_index(drop=True)


srcs = _load_geochemistry(DATA_DIR / "KRA21_sources.csv", "Name")
study = _load_geochemistry(DATA_DIR / "study_samples.csv", "Sample")

print(f"Data directory: {DATA_DIR}")
print(f"Source samples: {len(srcs)} across {srcs['Group'].nunique()} groups")
print(f"Study samples:  {len(study)} across {study['Group'].nunique()} groups")


Data directory: g:\Shared drives\Obsidian Geochem\visualization_nb\obsidian-source-anillo\data
Source samples: 294 across 7 groups
Study samples:  15 across 2 groups


In [13]:
def confidence_ellipse_points(x, y, scale=np.sqrt(-2 * np.log(0.05)), size=120):
    """Return the 95% bivariate normal data-spread ellipse by default."""
    values = np.column_stack((np.asarray(x, dtype=float), np.asarray(y, dtype=float)))
    if len(values) < 2:
        return None

    covariance = np.cov(values, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = np.clip(eigenvalues[order], 0, None)
    eigenvectors = eigenvectors[:, order]

    angles = np.linspace(0, 2 * np.pi, size)
    unit_circle = np.column_stack((np.cos(angles), np.sin(angles)))
    return unit_circle @ np.diag(scale * np.sqrt(eigenvalues)) @ eigenvectors.T + values.mean(axis=0)


groups = list(pd.unique(pd.concat([srcs["Group"], study["Group"]], ignore_index=True)))
name_to_color = {
    group: DEFAULT_PLOTLY_COLORS[index % len(DEFAULT_PLOTLY_COLORS)]
    for index, group in enumerate(groups)
}


In [14]:
import base64
import io
import json

import ipywidgets as widgets
from IPython.display import HTML, display
from matplotlib.figure import Figure
from matplotlib.patches import Polygon


def _show_plot(output, plot):
    output.outputs = ({
        "output_type": "display_data",
        "data": {"application/vnd.plotly.v1+json": json.loads(plot.to_json())},
        "metadata": {},
    },)


def _pdf_color(color):
    if isinstance(color, str) and color.startswith("rgb("):
        return tuple(float(value) / 255 for value in color[4:-1].split(","))
    return color


def _add_ellipse_label(fig, ellipse, label, color, legendgroup, ternary=False):
    center = ellipse[:-1].mean(axis=0)
    options = dict(mode="text", text=[label], textposition="middle center",
                   textfont=dict(color=color), showlegend=False,
                   legendgroup=legendgroup, hoverinfo="skip")
    if ternary:
        fig.add_trace(go.Scatterternary(a=[center[0]], b=[center[1]],
                                        c=[1 - center.sum()], **options))
    else:
        fig.add_trace(go.Scatter(x=[center[0]], y=[center[1]], **options))


def _plot_pdf(plot, ternary=False):
    fig = Figure(figsize=(10, 7))
    ax = fig.subplots()
    triangle = None
    if ternary:
        height = np.sqrt(3) / 2
        triangle = Polygon([(0, 0), (1, 0), (0.5, height)], closed=True,
                           fill=False, edgecolor="black")
        ax.add_patch(triangle)
        for fraction in np.arange(0.2, 1, 0.2):
            for x, y in (
                ([fraction / 2, 1 - fraction / 2], [height * fraction] * 2),
                ([1 - fraction, (1 - fraction) / 2], [0, height * (1 - fraction)]),
                ([fraction, (1 + fraction) / 2], [0, height * (1 - fraction)]),
            ):
                ax.plot(x, y, color="lightgrey", linewidth=0.5)
        ax.text(0.5, height + 0.04, plot.layout.ternary.aaxis.title.text, ha="center")
        ax.text(0, -0.05, plot.layout.ternary.baxis.title.text, ha="center")
        ax.text(1, -0.05, plot.layout.ternary.caxis.title.text, ha="center")
        ax.set_xlim(-0.08, 1.08)
        ax.set_ylim(-0.08, height + 0.08)
        ax.set_aspect("equal")
        ax.axis("off")
    else:
        ax.set_xlabel(plot.layout.xaxis.title.text)
        ax.set_ylabel(plot.layout.yaxis.title.text)

    for trace in plot.data:
        if ternary:
            a, b, c = (np.asarray(values, dtype=float) for values in (trace.a, trace.b, trace.c))
            x, y = c + 0.5 * a, height * a
        else:
            x, y = np.asarray(trace.x, dtype=float), np.asarray(trace.y, dtype=float)
        mode = trace.mode or ""
        if "lines" in mode:
            color = _pdf_color(trace.line.color)
            line, = ax.plot(x, y, color=color, linestyle="--" if trace.line.dash == "dash" else "-",
                            label=trace.name)
            if triangle is not None:
                line.set_clip_path(triangle)
        if "markers" in mode:
            color = _pdf_color(trace.marker.color)
            marker = "x" if trace.marker.symbol == "x" else "o"
            points = ax.scatter(x, y, color=color, marker=marker, s=30, label=trace.name)
            if triangle is not None:
                points.set_clip_path(triangle)
        if "text" in mode and trace.text is not None:
            color = _pdf_color(trace.textfont.color or trace.marker.color)
            offset = 6 if "markers" in mode else 0
            for x_value, y_value, text in zip(x, y, trace.text):
                ax.annotate(str(text), (x_value, y_value), xytext=(0, offset),
                            textcoords="offset points", ha="center", fontsize=7, color=color)
    ax.set_title(plot.layout.title.text)
    if plot.data:
        ax.legend(title="Source / study group", loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="pdf", bbox_inches="tight")
    return buffer.getvalue()


def make_export_button(plot_fn, filename_fn, ternary=False):
    button = widgets.Button(description="Export PDF", icon="download")
    output = widgets.Output()

    def export_pdf(_):
        with output:
            output.clear_output(wait=True)
            pdf = _plot_pdf(plot_fn(), ternary=ternary)
            encoded = base64.b64encode(pdf).decode("ascii")
            filename = filename_fn() + ".pdf"
            display(HTML(f'<a download="{filename}" href="data:application/pdf;base64,{encoded}">Download {filename}</a>'))

    button.on_click(export_pdf)
    return button, output


# Use the same candidate elements as the obsidian-andes biplot.
plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]
available_elements = [
    element for element in plot_elements
    if element in srcs.columns and element in study.columns
    and np.isfinite(srcs[element]).any() and np.isfinite(study[element]).any()
]
if not available_elements:
    raise ValueError("The source and study tables have no common usable elements to plot.")

x_menu = widgets.Dropdown(options=available_elements, value="Sr", description="X axis:")
y_menu = widgets.Dropdown(options=available_elements, value="Zr", description="Y axis:")
labels_button = widgets.ToggleButton(value=False, description="Show Labels", icon="tag")
biplot_output = widgets.Output()


def build_biplot(x_col, y_col, show_labels=False):
    fig = go.Figure()

    for group, source_group in srcs.groupby("Group", sort=False):
        color = name_to_color[group]
        valid = np.isfinite(source_group[x_col]) & np.isfinite(source_group[y_col])
        source_group = source_group.loc[valid]
        if source_group.empty:
            continue
        ellipse = confidence_ellipse_points(source_group[x_col], source_group[y_col])
        if ellipse is not None:
            fig.add_trace(go.Scatter(
                x=ellipse[:, 0], y=ellipse[:, 1], mode="lines",
                name=f"{group} source ellipse", legendgroup=str(group),
                line=dict(color=color), hoverinfo="skip",
            ))
            _add_ellipse_label(fig, ellipse, f"{group} (source)", color, str(group))
        else:
            fig.add_trace(go.Scatter(
                x=source_group[x_col], y=source_group[y_col], mode="markers",
                name=f"{group} source sample", legendgroup=str(group),
                text=source_group["Sample"],
                hovertemplate="Source sample: %{text}<extra></extra>",
                marker=dict(symbol="x", color=color, size=8),
            ))

    for group, study_group in study.groupby("Group", sort=False):
        color = name_to_color[group]
        valid = np.isfinite(study_group[x_col]) & np.isfinite(study_group[y_col])
        study_group = study_group.loc[valid]
        if study_group.empty:
            continue
        fig.add_trace(go.Scatter(
            x=study_group[x_col], y=study_group[y_col],
            mode="markers+text" if show_labels else "markers",
            name=f"Study: {group}", legendgroup=str(group),
            text=study_group["Sample"], textposition="top center",
            hovertemplate="Study sample: %{text}<extra></extra>",
            marker=dict(color=color, size=8),
        ))
        ellipse = confidence_ellipse_points(study_group[x_col], study_group[y_col])
        if ellipse is not None:
            fig.add_trace(go.Scatter(
                x=ellipse[:, 0], y=ellipse[:, 1], mode="lines",
                name=f"{group} study ellipse", legendgroup=str(group),
                line=dict(color=color, dash="dash"), hoverinfo="skip",
            ))
            _add_ellipse_label(fig, ellipse, f"{group} (study)", color, str(group))

    fig.update_layout(
        title="Study Samples and Known Obsidian Sources",
        xaxis_title=x_col,
        yaxis_title=y_col,
        template="plotly_white",
        legend_title="Source / study group",
    )
    return fig


def update_biplot(change=None):
    _show_plot(biplot_output, build_biplot(x_menu.value, y_menu.value, labels_button.value))


x_menu.observe(update_biplot, names="value")
y_menu.observe(update_biplot, names="value")
labels_button.observe(update_biplot, names="value")
biplot_export, biplot_export_output = make_export_button(
    lambda: build_biplot(x_menu.value, y_menu.value, labels_button.value),
    lambda: f"biplot_{x_menu.value}_{y_menu.value}",
)
display(labels_button, widgets.HBox([x_menu, y_menu]), biplot_output,
        biplot_export, biplot_export_output)
update_biplot()


ToggleButton(value=False, description='Show Labels', icon='tag')

Output()

Button(description='Export PDF', icon='download', style=ButtonStyle())

Output()

In [15]:
def _normalized_composition(frame, elements):
    values = frame[elements].to_numpy(dtype=float)
    totals = values.sum(axis=1)
    valid = np.isfinite(values).all(axis=1) & (values >= 0).all(axis=1) & (totals > 0)
    normalized = values[valid] / totals[valid, None]
    return frame.loc[valid].reset_index(drop=True), normalized


def build_ternary_plot(a_col, b_col, c_col, show_labels=False):
    elements = [a_col, b_col, c_col]
    ternary_sources, source_fractions = _normalized_composition(srcs, elements)
    ternary_study, study_fractions = _normalized_composition(study, elements)
    if len(ternary_sources) != len(srcs) or len(ternary_study) != len(study):
        print("Rows with invalid ternary compositions were omitted from the ternary diagram.")

    fig = go.Figure()
    for group in pd.unique(ternary_sources["Group"]):
        mask = ternary_sources["Group"].to_numpy() == group
        fractions = source_fractions[mask]
        color = name_to_color[group]
        if len(fractions) >= 2:
            ellipse = confidence_ellipse_points(fractions[:, 0], fractions[:, 1])
            if ellipse is not None:
                fig.add_trace(go.Scatterternary(
                    a=ellipse[:, 0], b=ellipse[:, 1], c=1 - ellipse[:, 0] - ellipse[:, 1],
                    mode="lines", name=f"{group} source ellipse",
                    legendgroup=f"{group} source ellipse",
                    line=dict(color=color), hoverinfo="skip",
                ))
                _add_ellipse_label(fig, ellipse, f"{group} (source)", color,
                                   f"{group} source ellipse", ternary=True)
        else:
            fig.add_trace(go.Scatterternary(
                a=fractions[:, 0], b=fractions[:, 1], c=fractions[:, 2],
                mode="markers", name=f"{group} source sample",
                text=ternary_sources.loc[mask, "Sample"],
                hovertemplate="Source sample: %{text}<extra></extra>",
                marker=dict(symbol="x", color=color, size=8),
            ))

    for group in pd.unique(ternary_study["Group"]):
        mask = ternary_study["Group"].to_numpy() == group
        fractions = study_fractions[mask]
        color = name_to_color[group]
        fig.add_trace(go.Scatterternary(
            a=fractions[:, 0], b=fractions[:, 1], c=fractions[:, 2],
            mode="markers+text" if show_labels else "markers", name=f"Study: {group}",
            text=ternary_study.loc[mask, "Sample"], textposition="top center",
            hovertemplate="Study sample: %{text}<extra></extra>",
            marker=dict(color=color, size=8),
        ))
        ellipse = confidence_ellipse_points(fractions[:, 0], fractions[:, 1])
        if ellipse is not None:
            fig.add_trace(go.Scatterternary(
                a=ellipse[:, 0], b=ellipse[:, 1], c=1 - ellipse[:, 0] - ellipse[:, 1],
                mode="lines", name=f"{group} study ellipse",
                legendgroup=f"{group} study ellipse",
                line=dict(color=color, dash="dash"), hoverinfo="skip",
            ))
            _add_ellipse_label(fig, ellipse, f"{group} (study)", color,
                               f"{group} study ellipse", ternary=True)

    fig.update_layout(
        title=f"Ternary Plot: {a_col}-{b_col}-{c_col}",
        ternary=dict(sum=1, aaxis=dict(title=a_col), baxis=dict(title=b_col), caxis=dict(title=c_col)),
        legend_title="Source / study group",
        legend=dict(x=1.02, y=1, xanchor="left", yanchor="top"),
    )
    return fig


if len(available_elements) < 3:
    raise ValueError("At least three common usable elements are needed for the ternary plot.")

ternary_a_menu = widgets.Dropdown(options=available_elements, value="Rb", description="A axis:")
ternary_b_menu = widgets.Dropdown(options=available_elements, value="Sr", description="B axis:")
ternary_c_menu = widgets.Dropdown(options=available_elements, value="Zr", description="C axis:")
ternary_output = widgets.Output()


def update_ternary(change=None):
    _show_plot(ternary_output, build_ternary_plot(
        ternary_a_menu.value, ternary_b_menu.value, ternary_c_menu.value,
        labels_button.value,
    ))


for menu in (ternary_a_menu, ternary_b_menu, ternary_c_menu):
    menu.observe(update_ternary, names="value")
labels_button.observe(update_ternary, names="value")
ternary_export, ternary_export_output = make_export_button(
    lambda: build_ternary_plot(ternary_a_menu.value, ternary_b_menu.value,
                               ternary_c_menu.value, labels_button.value),
    lambda: f"ternary_{ternary_a_menu.value}_{ternary_b_menu.value}_{ternary_c_menu.value}",
    ternary=True,
)
display(widgets.HBox([
    ternary_a_menu, ternary_b_menu, ternary_c_menu,
]), labels_button,
    ternary_output, ternary_export, ternary_export_output)
update_ternary()


ToggleButton(value=False, description='Show Labels', icon='tag')

Output()

Button(description='Export PDF', icon='download', style=ButtonStyle())

Output()

### Analysis complete

The biplot and ternary diagram use the checked-in source and study tables. Use each plot's dropdowns to change its axes and the Show Labels button to toggle study sample labels.